# Baseline Model

## Package Import & Path Settings

In [22]:
import os
import pandas as pd
from pandas import DataFrame
import numpy as np
import matplotlib.pyplot as plt
import json
import hashlib
from pathlib import Path
from collections import defaultdict

from PIL import Image, UnidentifiedImageError

# pytorch
from torchvision.datasets import ImageFolder
from torch.utils.data import Subset
from torchvision import transforms
from torch import nn
import torch

# data split
from sklearn.model_selection import train_test_split

# preprocessing & structure
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.compose import TransformedTargetRegressor
from sklearn.preprocessing import FunctionTransformer

# baseline models
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error

# others
from concurrent.futures import ThreadPoolExecutor
from src.dataset import load_filtered_imagefolder, rgba_to_rgb_with_bg

# path settings
from src import MODELS_DIR, PARAMS_PATH, SEED, PET_IMAGES_DIR, get_device
DEVICE = get_device()
print(f'Device: {DEVICE}')
print(f'MODELS_DIR: {MODELS_DIR}')
print(f'PARAMS_PATH: {PARAMS_PATH}')
print(f'SEED: {SEED}')
print(f'PET_IMAGES_DIR: {PET_IMAGES_DIR}')

# project constants
label = {0:'cat', 1:'dog'}
IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD  = (0.229, 0.224, 0.225)
SAMPLE_SIZE = 256


Device: cpu
MODELS_DIR: C:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\outputs\models
PARAMS_PATH: C:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\outputs\params
SEED: 37
PET_IMAGES_DIR: C:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\data\raw\PetImages


## Data Loading


In [8]:
train_tf = transforms.Compose([
    transforms.Resize(SAMPLE_SIZE),
    transforms.RandomCrop(SAMPLE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])
val_tf = transforms.Compose([
    transforms.Resize(SAMPLE_SIZE),
    transforms.CenterCrop(SAMPLE_SIZE),    
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD)
])

In [9]:
bad_path = PET_IMAGES_DIR / 'bad_files.json'
dup_group_path = PET_IMAGES_DIR / 'duplicate_groups_exact.json'
ignore_path = PET_IMAGES_DIR / 'ignore_files_list.json'

ds_tr, excl_set = load_filtered_imagefolder(PET_IMAGES_DIR, bad_path, dup_group_path, ignore_path, train_tf, rgba_to_rgb_with_bg)
ds_val, _ = load_filtered_imagefolder(PET_IMAGES_DIR, bad_path, dup_group_path, ignore_path, val_tf, rgba_to_rgb_with_bg)

print(f'Dataset Loaded from {PET_IMAGES_DIR}')
print(f'Number of valid samples: {len(ds_tr)}')
print(f'Number of samples excluded: {len(excl_set)}')

C:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\src\dataset.py:184: UserWarning: duplicate_groups missing total file count in C:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\data\raw\PetImages\duplicate_groups_exact.json. Please update this list.
  _check_total(dup_meta, "duplicate_groups", dup_path)


Dataset Loaded from C:\Coding Stuff\Python\kaggle\kaggle-image-classification-cats_dogs\data\raw\PetImages
Number of valid samples: 24968
Number of samples excluded: 32


### Test Data Split

Train / Test Data Split: 50%

In [ ]:
pd.DataFrame(ds_tr.targets).value_counts()


In [ ]:
ds_idx = np.array(range(len(ds_tr)))
X_train, X_test, y_train, y_test = train_test_split(ds_idx, ds_tr.)

['Cat', 'Dog']

In [ ]:
train_test_split()

In [75]:
?Subset

Init signature:
Subset(
    dataset: torch.utils.data.dataset.Dataset[+_T_co],
    indices: collections.abc.Sequence[int],
) -> None
Docstring:     
Subset of a dataset at specified indices.

Args:
    dataset (Dataset): The whole Dataset
    indices (sequence): Indices in the whole set selected for subset
File:           c:\coding stuff\python\kaggle\kaggle-image-classification-cats_dogs\.venv\lib\site-packages\torch\utils\data\dataset.py
Type:           type
Subclasses:     

## Baseline Model

A simple CNN model is constructed with PyTorch. The model consists of the following components:
1. **Feature Extraction**:
   1. Conv2D

In [68]:
class CNN_Clf(nn.Module):
    def __init__(self, num_class=2):
        super().__init__()
        self.feature = nn.Sequential(
            nn.Conv2d(3, 32, 5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 5, padding=2),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.classifier = nn.Linear(64, num_class)

        pass   

    def forward(self, x):
        x = self.feature(x)
        x = torch.flatten(x, 1)
        logits = self.classifier(x)
        return logits
    
    @ torch.no_grad()
    def predict(self, x):
        self.eval()
        return self.forward(x)
    
    @ torch.no_grad()
    def predict_proba(self, x):
        self.eval()
        return torch.softmax(self.forward(x), dim=1, dtype=torch.float32)
    
    @ torch.no_grad()
    def predict_id(self, x):
        self.eval()
        proba = self.predict_proba(x)
        ids = torch.argmax(proba, axis=1)       
        return ids


Smoke test result shows no shape/data type error during the inferencing.

In [74]:
a = np.random.rand(32, 3, 128, 128).astype('float32')
a_tr = torch.from_numpy(a).to(DEVICE)
cnn_clf = CNN_Clf(2).to(DEVICE)
logits = cnn_clf.predict(a_tr).to('cpu').detach().numpy()
print(logits[0:5])


[[-0.15117726  0.324121  ]
 [-0.15135403  0.3237161 ]
 [-0.15102679  0.322982  ]
 [-0.15093632  0.32195884]
 [-0.1507387   0.32320327]]


### Training Loop